# IMDB Sentiment Dataset — Exploratory Data Analysis

Before we train any model, we explore the data. This notebook answers:
- What does the data look like?
- Are the classes balanced?
- How long are the reviews?
- What words appear most often in positive vs negative reviews?

In [ ]:
from datasets import load_dataset
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import sys, os

sys.path.insert(0, os.path.join('..', 'backend'))
from preprocess import clean_text

sns.set_theme(style='whitegrid', palette='muted')

## 1. Load the dataset

In [ ]:
raw = load_dataset('imdb')
train_df = pd.DataFrame(raw['train'])
test_df  = pd.DataFrame(raw['test'])

# label 0 = Negative, label 1 = Positive
train_df['sentiment'] = train_df['label'].map({0: 'Negative', 1: 'Positive'})
print('Train size:', len(train_df))
print('Test size: ', len(test_df))
train_df.head(3)

## 2. Class balance

An imbalanced dataset (e.g. 90% positive) would make accuracy misleading.
IMDB is perfectly balanced — 50/50.

In [ ]:
counts = train_df['sentiment'].value_counts()
print(counts)

counts.plot(kind='bar', color=['#dc2626', '#16a34a'], edgecolor='white', figsize=(5,4))
plt.title('Class Distribution (Train)')
plt.ylabel('Number of Reviews')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 3. Review length distribution

Longer reviews carry more signal, but also cost more to process.

In [ ]:
train_df['word_count'] = train_df['text'].str.split().str.len()

print(train_df['word_count'].describe().round(1))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for sentiment, color, ax in zip(
    ['Positive', 'Negative'],
    ['#16a34a', '#dc2626'],
    axes
):
    subset = train_df[train_df['sentiment'] == sentiment]['word_count']
    subset.clip(upper=1000).plot(kind='hist', bins=40, color=color, ax=ax, edgecolor='white')
    ax.set_title(f'{sentiment} Review Lengths')
    ax.set_xlabel('Word count (capped at 1000)')

plt.tight_layout()
plt.show()

## 4. Word clouds — what words signal each class?

In [ ]:
def make_wordcloud(texts, color):
    cleaned = ' '.join(clean_text(t) for t in texts)
    wc = WordCloud(
        width=800, height=400,
        background_color='white',
        colormap='Greens' if color == 'green' else 'Reds',
        max_words=100
    ).generate(cleaned)
    return wc

pos_texts = train_df[train_df['sentiment'] == 'Positive']['text'].tolist()
neg_texts = train_df[train_df['sentiment'] == 'Negative']['text'].tolist()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.imshow(make_wordcloud(pos_texts, 'green'), interpolation='bilinear')
ax1.set_title('Positive Reviews', fontsize=14)
ax1.axis('off')

ax2.imshow(make_wordcloud(neg_texts, 'red'), interpolation='bilinear')
ax2.set_title('Negative Reviews', fontsize=14)
ax2.axis('off')

plt.tight_layout()
plt.show()

## 5. Sample reviews

Always read some raw examples — visualizations can hide quirks in the text.

In [ ]:
for _, row in train_df.groupby('sentiment').head(2).iterrows():
    print(f'--- {row["sentiment"]} ---')
    print(row['text'][:300])
    print()

## Key takeaways

- Dataset is perfectly balanced — no need for oversampling/undersampling
- Reviews contain HTML tags (`<br />`) — our `clean_text()` strips these
- Median review length ~230 words — short enough for TF-IDF, borderline for basic LSTMs
- Word clouds show clear signal: positive uses words like *great*, *love*, *best*; negative uses *bad*, *worst*, *boring*

**Next step:** run `backend/train.py` to train the model.